# ERA5 to PRISM Downscaling (Georgia)
Spatiotemporal downscaling from coarse ERA5 reanalysis to higher-resolution PRISM temperature fields.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "datasets").exists() and (ROOT.parent / "datasets").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project root: {ROOT}")


## 1. Problem Setup
- ERA5 provides large-scale atmospheric fields at coarse spatial resolution.
- PRISM provides finer regional temperature grids.
- The learning target is `ERA5(t-k+1 ... t) -> PRISM(t)`.


## 2. Modeling Approach
- Persistence baseline
- Global linear baseline
- CNN spatial baseline
- ConvLSTM temporal model

The comparison tests whether temporal context improves downscaling quality.

## 3. Multi-variable inputs
Using only temperature was limiting. Adding 10m wind (`u10`, `v10`) and surface pressure (`sp`) gives the model more context for regional structure.

## 4. Data Overview
- ERA5 NetCDF dimensions: time, latitude, longitude
- PRISM daily rasters: high-resolution target grids
- Dataset windows: input shape `[T, 4, H, W]`, target shape `[1, H_high, W_high]`


In [ ]:
import json
import pandas as pd
import xarray as xr
from IPython.display import Image, Markdown, display

from datasets.prism_dataset import ERA5_PRISM_Dataset

ERA5_PATH = ROOT / "data_raw/era5_georgia_temp.nc"
PRISM_DIR = ROOT / "data_raw/prism"
RESULTS_DIR = ROOT / "results/evaluation"

if ERA5_PATH.exists() and PRISM_DIR.exists():
    era5_ds = xr.open_dataset(ERA5_PATH)
    print("ERA5 variables:", list(era5_ds.data_vars))
    if era5_ds.data_vars:
        first_var = list(era5_ds.data_vars)[0]
        print(f"Sample ERA5 shape ({first_var}):", tuple(era5_ds[first_var].shape))

    dataset = ERA5_PRISM_Dataset(str(ERA5_PATH), str(PRISM_DIR), history_length=3, verbose=False)
    x, y = dataset[0]
    print("Usable samples:", len(dataset))
    print("Input shape [T, C, H, W]:", tuple(x.shape))
    print("Target shape [C, H, W]:", tuple(y.shape))
else:
    print("Data is missing. Run:")
    print("python data_pipeline/download_era5_georgia.py --year 2023 --month 1")
    print("python data_pipeline/download_prism.py --start-date 20230101 --days 10 --variable tmean")


## 5. Results
### Model Comparison Summary
- Linear baseline is strongest in this short run.
- CNN and ConvLSTM show expected behavior but need more temporal coverage for stable gains.


In [ ]:
cnn_images = sorted((RESULTS_DIR / "cnn").glob("comparison_*.png"))
convlstm_images = sorted((RESULTS_DIR / "convlstm").glob("comparison_*.png"))
summary_csv_path = RESULTS_DIR / "baselines_summary.csv"
model_comparison_path = RESULTS_DIR / "model_comparison.png"
cnn_metrics_path = RESULTS_DIR / "cnn" / "metrics.json"
convlstm_metrics_path = RESULTS_DIR / "convlstm" / "metrics.json"

if cnn_images:
    display(Markdown("### CNN output"))
    display(Image(filename=str(cnn_images[0])))
if convlstm_images:
    display(Markdown("### ConvLSTM output"))
    display(Image(filename=str(convlstm_images[0])))
if model_comparison_path.exists():
    display(Markdown("### Model comparison"))
    display(Image(filename=str(model_comparison_path)))

rows = []
for model_name, metrics_path in (("cnn", cnn_metrics_path), ("convlstm", convlstm_metrics_path)):
    if metrics_path.exists():
        metrics = json.loads(metrics_path.read_text())
        rows.append({
            "model": model_name,
            "rmse": metrics.get("rmse"),
            "mae": metrics.get("mae"),
            "bias": metrics.get("bias"),
            "num_samples": metrics.get("num_samples"),
        })
if rows:
    display(Markdown("### Per-model metrics"))
    display(pd.DataFrame(rows))

if summary_csv_path.exists():
    display(Markdown("### Baseline summary table"))
    display(pd.read_csv(summary_csv_path))

missing = (
    not cnn_images
    or not convlstm_images
    or not summary_csv_path.exists()
    or not model_comparison_path.exists()
)
if missing:
    print("Run:")
    print("python training/train_downscaler.py --model cnn --history-length 1 --epochs 5")
    print("python training/train_downscaler.py --model convlstm --history-length 3 --epochs 5")
    print("python training/train_downscaler.py --model cnn --history-length 3 --epochs 5")
    print("python evaluation/evaluate_model.py --models persistence linear cnn convlstm --history-length 3 --num-samples 8")


## 6. Metrics Interpretation
- RMSE and MAE measure average pixel-level temperature error. Lower is better.
- In this run, linear baseline is best.
- ConvLSTM is currently limited by short training span and sample count.


## 7. Visual Analysis
- ERA5 input is smooth and coarse.
- PRISM target has finer local variability.
- CNN captures broad structure but smooths sharp features.
- ConvLSTM adds temporal context but still needs more data to close the resolution gap.


## 8. Limitations
- short temporal span in current run
- one region (Georgia)
- only four ERA5 predictors
- model scale is small compared with large operational weather systems


## 9. Relation to Existing Work
ConvLSTM is a standard spatiotemporal baseline. The pipeline follows the same temporal-conditioning direction used in modern weather ML, but stays focused on a tractable regional downscaling setup.

## 10. Future Work
- longer temporal windows
- additional ERA5 variables
- multi-source inputs (remote sensing or high-resolution imagery)
- uncertainty-aware prediction heads
